<H1> Feature Engineering </H1>

This notebook transforms raw creature data into engineered features for HP prediction.

**Input:** `data/dnd5e_monsters_from_json.csv`  
**Output:** `helper_files/engineered_features.parquet`

# Imports and Configs

In [1]:
import json
import numpy as np
import os
import pandas as pd

import sys


In [2]:
print(sys.executable)

/workspaces/matrix_v0/.venv/bin/python


In [3]:
# %pip install pyarrow
# %pip install fastparquet

In [4]:
# Detect execution context and set paths dynamically
from pathlib import Path

# Get the current working directory
cwd = Path.cwd()

# Check if we're in the notebooks directory or project root
if cwd.name == 'notebooks':
    # Running from notebooks directory (in Jupyter)
    DATA_DIR = '../data'
    PICKLED_MODELS_DIR = '../pickled_models'
    MONSTER_BUILDER_DIR = '../monster-builder-v2'
    HELPERS_DIR = './helper_files'
    IN_NB_DIR = False
else:
    # Running from project root (via run_three_tier_model.py)
    DATA_DIR = './data'
    PICKLED_MODELS_DIR = './pickled_models'
    MONSTER_BUILDER_DIR = './monster-builder-v2'
    HELPERS_DIR = './notebooks/helper_files'
    IN_NB_DIR = False

print(f"📁 Execution context detected:")
print(f"   Current directory: {cwd}")
print(f"   Data directory: {DATA_DIR}")
print(f"   Models directory: {PICKLED_MODELS_DIR}")


📁 Execution context detected:
   Current directory: /workspaces/matrix_v0
   Data directory: ./data
   Models directory: ./pickled_models


In [5]:

# Add helper_files to path
# sys.path.insert(0, '.')
if IN_NB_DIR is True:
    from helper_files import (
        # Config
        CONDITIONS, PHASE2_FEATURES, PHASE2_PENALTIES, get_cr_tier, get_phase3_features,
        # Parsers
        parse_cr, parse_hp, parse_ac, parse_speed, SIZE_ORDINAL_MAP,
        count_proficiencies, has_sense, parse_sense_range, parse_passive_perception,
        count_abilities, parse_legendary_actions, parse_attack_bonus, parse_save_dc,
        parse_dpr_from_json, parse_charge_bonus_attack, parse_legendary_actions_dpr,
        parse_legendary_conditions, extract_spellcaster_level,
        has_advantage_condition, has_disadvantage_condition, has_attackers_advantage,
        get_resistance_multiplier, get_immunity_multiplier, extract_family,
        # Baselines
        get_baseline_hp, get_baseline_ac, get_baseline_attack, get_baseline_dpr,
        get_baseline_dc, get_baseline_size_ordinal, get_baseline_speed_ground,
        get_fly_speed_baseline, get_darkvision_baseline,
    )

    print("Imports successful")
else:
    from notebooks.helper_files import (
    # Config
    CONDITIONS, PHASE2_FEATURES, PHASE2_PENALTIES, get_cr_tier, get_phase3_features,
    # Parsers
    parse_cr, parse_hp, parse_ac, parse_speed, SIZE_ORDINAL_MAP,
    count_proficiencies, has_sense, parse_sense_range, parse_passive_perception,
    count_abilities, parse_legendary_actions, parse_attack_bonus, parse_save_dc,
    parse_dpr_from_json, parse_charge_bonus_attack, parse_legendary_actions_dpr,
    parse_legendary_conditions, extract_spellcaster_level,
    has_advantage_condition, has_disadvantage_condition, has_attackers_advantage,
    get_resistance_multiplier, get_immunity_multiplier, extract_family,
    # Baselines
    get_baseline_hp, get_baseline_ac, get_baseline_attack, get_baseline_dpr,
    get_baseline_dc, get_baseline_size_ordinal, get_baseline_speed_ground,
    get_fly_speed_baseline, get_darkvision_baseline,
    )
    print("Imports successful")

Imports successful


# Load Raw Data

In [6]:
# Load main creature data
df = pd.read_csv(f'{DATA_DIR}/dnd5e_monsters_from_json.csv')
print(f"Loaded {len(df)} monsters")
print(f"Columns: {list(df.columns)[:10]}...")

Loaded 382 monsters
Columns: ['Name', 'Size', 'Type', 'Alignment', 'HP', 'AC', 'Speed', 'Challenge_Rating', 'XP', 'STR']...


# Actual Feature Engineering

## Parse Basic Features

In [7]:
# Parse CR
df['cr_numeric'] = df['Challenge_Rating'].apply(parse_cr)

# Parse HP
df['actual_hp'] = df['HP'].apply(parse_hp)

# Parse AC
df['ac_value'] = df['AC'].apply(parse_ac)

print(f"CR range: {df['cr_numeric'].min()} - {df['cr_numeric'].max()}")
print(f"HP range: {df['actual_hp'].min()} - {df['actual_hp'].max()}")

CR range: 0.0 - 30.0
HP range: 1 - 676


In [8]:
# Parse speeds
df['speed_ground'] = df['Speed'].apply(lambda x: parse_speed(x, 'ground'))
df['speed_fly'] = df['Speed'].apply(lambda x: parse_speed(x, 'fly'))
df['speed_swim'] = df['Speed'].apply(lambda x: parse_speed(x, 'swim'))
df['speed_burrow'] = df['Speed'].apply(lambda x: parse_speed(x, 'burrow'))
df['speed_climb'] = df['Speed'].apply(lambda x: parse_speed(x, 'climb'))

df['max_speed'] = df[['speed_ground', 'speed_fly', 'speed_swim', 'speed_burrow', 'speed_climb']].max(axis=1)
df['movement_types_count'] = (df[['speed_ground', 'speed_fly', 'speed_swim', 'speed_burrow', 'speed_climb']] > 0).sum(axis=1)
df['has_flying'] = (df['speed_fly'] > 0).astype(int)

# Parse size
df['size_ordinal'] = df['Size'].map(SIZE_ORDINAL_MAP).fillna(2)

print("Speeds and size parsed")

Speeds and size parsed


In [9]:
# Parse proficiencies
df['save_proficiency_count'] = df['Saving_Throws'].apply(count_proficiencies)
df['skill_proficiency_count'] = df['Skills'].apply(count_proficiencies)
df['resistance_count'] = df['Resistances'].apply(count_proficiencies)
df['immunity_count'] = df['Immunities'].apply(count_proficiencies)
df['vulnerability_count'] = df['Vulnerabilities'].apply(count_proficiencies)
df['condition_immunity_count'] = df['Condition_Immunities'].apply(count_proficiencies)

print("Proficiencies parsed")

Proficiencies parsed


In [10]:
# Parse senses
df['has_darkvision'] = df['Senses'].apply(lambda x: has_sense(x, 'darkvision'))
df['darkvision_range'] = df['Senses'].apply(lambda x: parse_sense_range(x, 'darkvision'))
df['has_blindsight'] = df['Senses'].apply(lambda x: has_sense(x, 'blindsight'))
df['has_truesight'] = df['Senses'].apply(lambda x: has_sense(x, 'truesight'))
df['has_tremorsense'] = df['Senses'].apply(lambda x: has_sense(x, 'tremorsense'))
df['passive_perception'] = df['Senses'].apply(parse_passive_perception)

print("Senses parsed")

Senses parsed


In [11]:
# Parse ability counts
df['trait_count'] = df['Traits'].apply(count_abilities)
df['action_count'] = df['Actions'].apply(count_abilities)
df['reaction_count'] = df['Reactions'].apply(count_abilities)
df['bonus_action_count'] = df['Bonus_Actions'].apply(count_abilities) if 'Bonus_Actions' in df.columns else 0

# Legendary actions
df[['has_legendary_actions', 'legendary_action_count', 'legendary_actions_per_round']] = df['Legendary_Actions'].apply(
    lambda x: pd.Series(parse_legendary_actions(x))
)

df['total_ability_count'] = df['trait_count'] + df['action_count'] + df['reaction_count'] + df['legendary_action_count']

print("Ability counts parsed")

Ability counts parsed


## Parse Combat Stats

In [12]:
# Parse attack bonus
df['highest_attack_bonus'] = df['Actions'].apply(parse_attack_bonus)

# Parse save DC
combined_abilities = (df['Traits'].fillna('') + ' ' + df['Actions'].fillna('') + ' ' +
                     df['Reactions'].fillna('') + ' ' + df['Legendary_Actions'].fillna(''))
df['highest_save_dc'] = combined_abilities.apply(parse_save_dc)

# DC Overrides (creatures with thematically high DCs)
DC_OVERRIDES = {
    'Green Hag': 14,  # DC 20 is for Illusory Appearance (thematic, not combat)
}
for creature, dc in DC_OVERRIDES.items():
    df.loc[df['Name'] == creature, 'highest_save_dc'] = dc

print(f"Applied {len(DC_OVERRIDES)} DC overrides")

Applied 1 DC overrides


In [13]:
# Parse DPR
df['estimated_dpr'] = df['Actions'].apply(parse_dpr_from_json)

# Add charge/pounce bonus DPR
df['charge_bonus_dpr'] = df.apply(lambda row: parse_charge_bonus_attack(row['Traits'], row['Actions']), axis=1)
df['estimated_dpr'] = df['estimated_dpr'] + df['charge_bonus_dpr']

# Parse legendary DPR
df['legendary_dpr'] = df['Legendary_Actions'].apply(parse_legendary_actions_dpr)
df['total_dpr'] = df['estimated_dpr'] + df['legendary_dpr']

print(f"DPR range: {df['total_dpr'].min():.1f} - {df['total_dpr'].max():.1f}")

DPR range: 0.0 - 148.0


## Parse Special Traits

In [14]:
# Special traits from combined abilities
df['has_legendary_resistance'] = combined_abilities.str.contains('legendary resistance', case=False, na=False).astype(int)
df['has_magic_resistance'] = combined_abilities.str.contains('magic resistance', case=False, na=False).astype(int)
df['has_regeneration'] = combined_abilities.str.contains('regeneration', case=False, na=False).astype(int)
df['has_spellcasting'] = combined_abilities.str.contains('spellcasting', case=False, na=False).astype(int)
df['spellcaster_level'] = combined_abilities.apply(extract_spellcaster_level)
df['has_grapple'] = combined_abilities.str.contains('grapple|grappled', case=False, na=False).astype(int)

print("Special traits parsed")

Special traits parsed


In [15]:
# Parse legendary conditions
df['legendary_conditions'] = df['Legendary_Actions'].apply(parse_legendary_conditions)

# Condition infliction features
for condition in CONDITIONS:
    feature_name = f'inflicts_{condition}'
    df[feature_name] = combined_abilities.str.contains(condition, case=False, na=False).astype(int)
    # Also check legendary actions
    df[feature_name] = df.apply(
        lambda row: 1 if (row[feature_name] == 1 or condition in row['legendary_conditions']) else 0,
        axis=1
    )

# Inflicts prone (Phase 2 feature)
df['inflicts_prone'] = combined_abilities.str.contains('prone', case=False, na=False).astype(int)
df['inflicts_prone'] = df.apply(
    lambda row: 1 if (row['inflicts_prone'] == 1 or 'prone' in row['legendary_conditions']) else 0,
    axis=1
)

print(f"{len(CONDITIONS)} condition features added")
print(f"Creatures that inflict prone: {df['inflicts_prone'].sum()} ({df['inflicts_prone'].mean()*100:.1f}%)")

10 condition features added
Creatures that inflict prone: 70 (18.3%)


In [16]:
# Advantage/disadvantage conditions
df['has_advantage_condition'] = df.apply(has_advantage_condition, axis=1)
df['has_disadvantage_condition'] = df.apply(has_disadvantage_condition, axis=1)
df['has_attackers_advantage'] = df.apply(has_attackers_advantage, axis=1)

print(f"Advantage conditions: {df['has_advantage_condition'].sum()} ({df['has_advantage_condition'].mean()*100:.1f}%)")
print(f"Disadvantage conditions: {df['has_disadvantage_condition'].sum()} ({df['has_disadvantage_condition'].mean()*100:.1f}%)")

Advantage conditions: 40 (10.5%)
Disadvantage conditions: 10 (2.6%)


## Calculate Baselines and Deviations

In [17]:
# Calculate baselines
df['hp_baseline'] = df['cr_numeric'].apply(get_baseline_hp)
df['ac_baseline'] = df['cr_numeric'].apply(get_baseline_ac)
df['attack_baseline'] = df['cr_numeric'].apply(get_baseline_attack)
df['dpr_baseline'] = df['cr_numeric'].apply(get_baseline_dpr)
df['dc_baseline'] = df['cr_numeric'].apply(get_baseline_dc)
df['size_ordinal_baseline'] = df['cr_numeric'].apply(get_baseline_size_ordinal)
df['speed_ground_baseline'] = df['cr_numeric'].apply(get_baseline_speed_ground)

# Calculate fly speed baseline (only for flyers)
df['fly_speed_baseline'] = df['cr_numeric'].apply(get_fly_speed_baseline)
df['speed_fly_deviation'] = df.apply(
    lambda row: row['speed_fly'] - row['fly_speed_baseline'] if row['speed_fly'] > 0 else 0,
    axis=1
)

# Calculate darkvision baseline (only for creatures with darkvision)
df['darkvision_baseline'] = df['cr_numeric'].apply(get_darkvision_baseline)
df['darkvision_deviation'] = df.apply(
    lambda row: row['darkvision_range'] - row['darkvision_baseline'] if row['darkvision_range'] > 0 else 0,
    axis=1
)

print("Baselines calculated")

Baselines calculated


In [18]:
# Calculate deviations
df['ac_deviation'] = df['ac_value'] - df['ac_baseline']
df['attack_deviation'] = df['highest_attack_bonus'] - df['attack_baseline']
df['dpr_deviation'] = df['total_dpr'] - df['dpr_baseline']
df['save_dc_deviation'] = df['highest_save_dc'] - df['dc_baseline']
df['size_ordinal_deviation'] = df['size_ordinal'] - df['size_ordinal_baseline']
df['speed_ground_deviation'] = df['speed_ground'] - df['speed_ground_baseline']

print("Deviations calculated")

Deviations calculated


## Phase 1.5: Resistance/Immunity Penalties

In [19]:
# Filter to valid HP
df_valid = df[df['actual_hp'] > 0].copy()
print(f"Valid samples: {len(df_valid)} monsters with HP > 0")

# Phase 1: Baseline HP
df_valid['hp_after_phase1'] = df_valid['hp_baseline']

# Calculate resistance/immunity penalties
df_valid['resistance_multiplier'] = df_valid['cr_numeric'].apply(get_resistance_multiplier)
df_valid['immunity_multiplier'] = df_valid['cr_numeric'].apply(get_immunity_multiplier)

df_valid['resistance_penalty'] = (
    df_valid['resistance_multiplier'] * 
    df_valid['hp_after_phase1'] * 
    (df_valid['resistance_count'] > 0)
)
df_valid['immunity_penalty'] = (
    df_valid['immunity_multiplier'] * 
    df_valid['hp_after_phase1'] * 
    (df_valid['immunity_count'] > 0)
)

df_valid['total_defensive_penalty'] = df_valid['immunity_penalty'] + df_valid['resistance_penalty']

# Apply 75% cap
df_valid['total_defensive_penalty'] = df_valid['total_defensive_penalty'].clip(
    upper=0.75 * df_valid['hp_after_phase1']
)

df_valid['hp_after_phase1_5'] = df_valid['hp_after_phase1'] - df_valid['total_defensive_penalty']

print("Phase 1.5 complete")

Valid samples: 382 monsters with HP > 0
Phase 1.5 complete


## Split by CR and Apply Phase 2 Penalties

In [20]:
# Split by CR tier
df_cr1 = df_valid[df_valid['cr_numeric'] < 1.0].copy()
df_cr2 = df_valid[(df_valid['cr_numeric'] >= 1.0) & (df_valid['cr_numeric'] <= 4.0)].copy()
df_cr3 = df_valid[(df_valid['cr_numeric'] >= 5.0) & (df_valid['cr_numeric'] <= 10.0)].copy()
df_cr4 = df_valid[(df_valid['cr_numeric'] >= 11.0) & (df_valid['cr_numeric'] <= 16.0)].copy()
df_cr5 = df_valid[df_valid['cr_numeric'] > 16.0].copy()

print(f"CR < 1:    {len(df_cr1)} monsters")
print(f"CR 1-4:    {len(df_cr2)} monsters")
print(f"CR 5-10:   {len(df_cr3)} monsters")
print(f"CR 11-16:  {len(df_cr4)} monsters")
print(f"CR > 16:   {len(df_cr5)} monsters")

CR < 1:    142 monsters
CR 1-4:    124 monsters
CR 5-10:   67 monsters
CR 11-16:  28 monsters
CR > 16:   21 monsters


In [21]:
def apply_phase2_penalties(df_tier, tier_key):
    """Apply Phase 2 penalties to a CR tier dataframe."""
    penalties = PHASE2_PENALTIES[tier_key]
    
    df_tier['hp_after_phase2'] = df_tier['hp_after_phase1_5'].copy()
    for feature, penalty in penalties.items():
        if feature in df_tier.columns:
            df_tier['hp_after_phase2'] += df_tier[feature] * penalty
    
    # Calculate scaled features
    df_tier['has_legendary_resistance_scaled'] = df_tier['has_legendary_resistance'] * df_tier['hp_after_phase2']
    df_tier['has_magic_resistance_scaled'] = df_tier['has_magic_resistance'] * df_tier['hp_after_phase2']
    df_tier['has_regeneration_scaled'] = df_tier['has_regeneration'] * df_tier['hp_after_phase2']
    
    # Calculate residual HP
    df_tier['residual_hp'] = df_tier['actual_hp'] - df_tier['hp_after_phase2']
    
    return df_tier

# Apply Phase 2 to each tier
df_cr1 = apply_phase2_penalties(df_cr1, 'cr1')
df_cr2 = apply_phase2_penalties(df_cr2, 'cr2')
df_cr3 = apply_phase2_penalties(df_cr3, 'cr3')
df_cr4 = apply_phase2_penalties(df_cr4, 'cr4')
df_cr5 = apply_phase2_penalties(df_cr5, 'cr5')

print("Phase 2 penalties applied to all tiers")

Phase 2 penalties applied to all tiers


In [22]:
# Add family and cr_tier for train/test splitting
for df_tier, tier_key in [(df_cr1, 'cr1'), (df_cr2, 'cr2'), (df_cr3, 'cr3'), (df_cr4, 'cr4'), (df_cr5, 'cr5')]:
    df_tier['family'] = df_tier['Name'].apply(extract_family)
    df_tier['cr_tier'] = tier_key

# Combine and Save

In [23]:
# Combine all tiers back together
df_engineered = pd.concat([df_cr1, df_cr2, df_cr3, df_cr4, df_cr5], ignore_index=True)
df_engineered = df_engineered.sort_values('cr_numeric')

print(f"Total engineered samples: {len(df_engineered)}")

Total engineered samples: 382


In [24]:
# Save to parquet
output_path = HELPERS_DIR + '/engineered_features.parquet'
df_engineered.to_parquet(output_path, index=False)

print(f"Saved {len(df_engineered)} monsters with {len(df_engineered.columns)} features")
print(f"Output: {output_path}")

Saved 382 monsters with 129 features
Output: ./notebooks/helper_files/engineered_features.parquet


In [25]:
# Summary of key features
phase3_features = get_phase3_features()
print(f"\nPhase 2 features: {len(PHASE2_FEATURES)}")
print(f"Phase 3 features: {len(phase3_features)}")
print(f"\nSample data:")
df_engineered[['Name', 'cr_numeric', 'actual_hp', 'hp_baseline', 'hp_after_phase1_5', 'hp_after_phase2', 'residual_hp']].head(10)


Phase 2 features: 9
Phase 3 features: 38

Sample data:


,Name,cr_numeric,actual_hp,hp_baseline,hp_after_phase1_5,hp_after_phase2,residual_hp
3,Animated Object (Large),0.0,20,4.5,3.375,16.875,3.125
2,Animated Object (Huge),0.0,40,4.5,3.375,12.375,27.625
5,Animated Object (Tiny),0.0,10,4.5,3.375,20.250,-10.250
4,Animated Object (Small or Medium),0.0,10,4.5,3.375,20.250,-10.250
7,Avatar of Death,0.0,1,4.5,3.375,26.375,-25.375
8,Awakened Shrub,0.0,10,4.5,3.375,40.750,-30.750
45,Giant Fire Beetle,0.0,4,4.5,4.500,35.125,-31.125
48,Giant Insect,0.0,30,4.5,4.500,9.500,20.500
61,Goat,0.0,4,4.5,4.500,0.625,3.375
92,Piranha,0.0,1,4.5,4.500,29.750,-28.750
